# Live Causal Debugging, Qwen3-4B-Instruct-2507

**Real open-weight model evaluation, not a simulation.** The code, 16 executable counterfactual bugs, recorded non-LLM control results, and patch-checking harness are embedded in this notebook. No GitHub repository or uploaded ZIP is required.

Use **Runtime > Change runtime type > T4 GPU** (or any GPU with enough memory). Running this notebook downloads and loads the official Hugging Face [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) model, executes the 16 real bug projects, runs the agent in five evidence conditions, applies patches and tests them.

**Model scores are not prepopulated.** A complete run is 16 bugs x 5 conditions = **80 generated model decisions**, plus conditional retrieval responses. The notebook may take a while on a T4 GPU; GPU availability is managed by Google Colab.

In [ ]:
!pip -q install "transformers>=4.51.0" "accelerate>=1.2" "bitsandbytes>=0.45" "huggingface_hub>=1.2" "pytest>=8" "safetensors>=0.5"

In [ ]:
# Restore executable study files from the embedded, compressed project snapshot.
import base64,io,zipfile,os
archive=base64.b64decode("...")
root='/content/DebugEvidenceBlindSpot'
os.makedirs(root,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    z.extractall(root)
os.chdir(root)
print('Extracted study files:',len(zipfile.ZipFile(io.BytesIO(archive)).namelist()))
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime, then restart the notebook. No Qwen result can be obtained without GPU here.')

### Verify the provided local control results
These files record executed non-LLM rules; they are *not* generated by Qwen.

In [ ]:
!PYTEST_DISABLE_PLUGIN_AUTOLOAD=1 python -m pytest -q -p no:cacheprovider tests/test_live_harness.py

### Real Qwen pilot (4 bug worlds, 20 model decisions)
Run the small pilot first. Every output records the raw model text, tokens, resolved model SHA, patch validation, and decision.

In [ ]:
!python live_eval.py --backend qwen --max-families 2 --repeat 3 --output results/live_qwen_pilot.jsonl

### Full model evaluation (16 bug worlds, 80 model decisions)
Run the complete experiment. This is the file required by the publication step.

In [ ]:
!python live_eval.py --backend qwen --repeat 3 --output results/live_qwen_all.jsonl

In [ ]:
!python verify_model_results.py results/live_qwen_all.jsonl --save results/live_qwen_metrics.json

### Optional ablation: evidence-aware system prompt
The primary Qwen run above uses a normal debugging instruction. This additional run tests whether explicitly discouraging unsupported guesses changes agent decisions.

In [ ]:
!python live_eval.py --backend qwen --prompt-mode evidence_aware --repeat 3 --output results/live_qwen_guardrail.jsonl

### Download actual generations and measurements
This ZIP includes the generated Qwen outputs and local controls. Keep the raw results, especially incorrect predictions, rather than reporting only summary numbers.

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile
with zipfile.ZipFile('/content/DebugEvidenceBlindSpot_Qwen_Measured.zip','w',zipfile.ZIP_DEFLATED) as z:
    for path in Path(root).rglob('*'):
        if path.is_file() and '.git' not in path.parts and '__pycache__' not in path.parts and '.pytest_cache' not in path.parts:
            z.write(path,path.relative_to(root))
files.download('/content/DebugEvidenceBlindSpot_Qwen_Measured.zip')

### Optional: publish the public Hugging Face Bucket
Only do this after all 80 real model results pass verification. Sign in to your HF account using the prompted secure mechanism. The bucket is public; GitHub is not modified.

In [ ]:
from huggingface_hub import login
login()  # Do not paste your access token into the notebook source or share notebook outputs.

In [ ]:
!python publish_live_bucket.py --bucket-name debug-evidence-blindspot-live